# 3. Heterogeneity Computations
The goal in this notebook is to generate estimates of heterogeneity by computing $I^2$, the proportion between-study variance due to heterogeneity rather than random sampling error. We compute this via simulation. Should be run after the Stan model is run, but before the visuals are generated.

In [1]:
library(tidyverse)
library(ggplot2)
library(rstan)
library(Hmisc)
set.seed(20)

Warning message:
"package 'tidyverse' was built under R version 4.3.3"
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: StanHeaders


rstan version 2.32.5 (Stan version 2.32.2)


For execution on a local, multicore CPU with excess RAM we recommend calling
options(mc.cores = parallel::detectCores()).
To avoid recompilation of unchanged Stan programs, we recommend calling
rstan_options(auto_write = TRUE)
For within-chain threading using `reduce_sum()` or `map_rect()` Stan functions,
change `threa

In [2]:
inv_logit = plogis
logit = qlogis

In [3]:
base_dir = getwd()
io_set = "Main Results"
model_input_dir  = file.path(base_dir, "Processed Data", io_set)
model_output_dir = file.path(base_dir, "Model Output", io_set)
output_dir = file.path(base_dir, "Heterogeneity Estimates", io_set)

#sev_class_type = "1997type"
#sev_class_type = "2009type"
sev_class_type = "hospitalisation"
save_output = TRUE

In [4]:
#Read in the model input data (which gives information on outcomes and scenarios)
input_data = file.path(model_input_dir, paste0("data_", sev_class_type, ".rds")) %>% readRDS

#Read in the model results. We are primarily interested in the pooled estimates p and the outcome estimates theta
model_results = file.path(model_output_dir, paste0("Results_LogisticRegression_mean=0_sd=2_sd_mean=0.5_sdsd=2_", sev_class_type, ".rds")) %>% readRDS

In [5]:
#Extract some variables from input data to get correct indices on things
scenario_df = input_data$scenario_df
outcome_df = input_data$outcome_df %>% mutate(ThetaIndex = 1:nrow(.))

#Get the reference region and the non-reference one
ref_region = input_data$ref_region
non_ref_region = ifelse(ref_region == "Asia", "Americas", "Asia")
ref_serotype_exposure = input_data$ref_serotype_exposure #Get reference serotype-prior exposure combination
ref_scenario = input_data$ref_scenario #Get reference scenario
non_ref_seroprior = input_data$char_mat_guide %>% filter(CharMatIndex >0) %>% pull(SeroPriorExp) %>% as.character #Get all non-reference serotype-prior exposures

#Get scenario labels ordered according to the model output which gives the reference scenario, then all non-reference serotype prior exposures in the reference region,
#followed by the reference serotype-exposure in the non-reference region, and the rest of the serotype-prior exposures in the non-reference region
scenario_labels = c(ref_scenario, 
                    paste0(ref_region, "-", non_ref_seroprior),
                    paste0(non_ref_region, "-", c(ref_serotype_exposure, non_ref_seroprior))
                   )

#Add these indices to the scenario DataFrame from the model input
scenario_p_matcher = data.frame(Scenario = scenario_labels) %>% mutate(ScenarioIndex = 1:nrow(.))
scenario_df = scenario_df %>% left_join(scenario_p_matcher, by = "Scenario")
scenario_df

SeroPriorExp,Region,Scenario,NumStudies,Severe,NonSevere,N,Inclusion,ScenIndex,CharMatIndex,ScenarioIndex
<fct>,<fct>,<chr>,<int>,<dbl>,<dbl>,<dbl>,<chr>,<int>,<dbl>,<int>
Unknown-DENV1,Americas,Americas-Unknown-DENV1,7,7496,17413,24909,Included,1,1,11
Unknown-DENV2,Americas,Americas-Unknown-DENV2,6,1326,2165,3491,Included,2,2,12
Unknown-DENV3,Americas,Americas-Unknown-DENV3,3,1770,2555,4325,Included,3,3,13
Unknown-DENV4,Americas,Americas-Unknown-DENV4,2,2011,6218,8229,Included,4,4,14
Primary-DENV1,Americas,Americas-Primary-DENV1,0,0,0,0,Excluded,-1,5,15
Primary-DENV2,Americas,Americas-Primary-DENV2,0,0,0,0,Excluded,-1,6,16
Primary-DENV3,Americas,Americas-Primary-DENV3,0,0,0,0,Excluded,-1,7,17
Primary-DENV4,Americas,Americas-Primary-DENV4,0,0,0,0,Excluded,-1,NA,NA
Secondary-DENV1,Americas,Americas-Secondary-DENV1,0,0,0,0,Excluded,-1,8,18


In [6]:
#We only generate I^2 estimates for included scenarios (those with at least 2 studies contributing data)
included_scenarios = scenario_df %>% filter(Inclusion == "Included")

#Get the available posterior samples for p and theta
p_est_draws = extract(model_results, pars = "p")$p
theta_est_draws = extract(model_results, pars = "theta")$theta

In [7]:
theta_est_draws

0.0220361567,0.473827967,0.2179110,0.4776241,0.5447827,0.3487387,0.0363741593,0.35188895,0.1177806,0.2715002,⋯,0.2150550,0.1703246,0.3536027,0.008603057,0.35730678,0.025760248,0.009493648,6.174700e-02,0.023156684,0.025899015
0.0054353554,0.063251391,0.2104748,0.4608320,0.5919819,0.3482950,0.0693941593,0.14170547,0.1204924,0.2614073,⋯,0.3105355,0.1963598,0.2216009,0.115559157,0.04659935,0.042765308,0.083890749,5.083315e-02,0.056981158,0.050350290
0.0343190383,0.142419577,0.2333995,0.4196006,0.5088506,0.3145338,0.0108766049,0.13152568,0.1175414,0.2480286,⋯,0.2182407,0.1587239,0.3685037,0.112336071,0.45636480,0.045157821,0.021251678,6.089574e-02,0.040377396,0.054358025
0.0729926082,0.084301617,0.2201977,0.3189805,0.5707413,0.3402949,0.1440922437,0.12871236,0.1156797,0.2535150,⋯,0.2283157,0.1818796,0.2685083,0.330186814,0.13544909,0.046089487,0.022390183,4.790156e-02,0.002912377,0.040327749
0.0116742426,0.013820919,0.2250310,0.3442469,0.5201749,0.3294308,0.0309044817,0.14303889,0.1091507,0.2669381,⋯,0.1473206,0.1881287,0.4111295,0.003624510,0.23149911,0.050658101,0.077278960,3.545835e-04,0.012865653,0.051937365
0.0733502214,0.159876584,0.1783119,0.4263967,0.5758592,0.3219949,0.0695305898,0.14079924,0.1184680,0.2320678,⋯,0.2052843,0.1688061,0.3499586,0.022675778,0.09020836,0.094180541,0.070992222,2.972635e-02,0.022140982,0.056566050
0.0071231738,0.058966517,0.1986779,0.4711850,0.6660963,0.3672456,0.0489570516,0.07123082,0.1158916,0.2454942,⋯,0.1847117,0.1929242,0.4036221,0.014273844,0.08356558,0.078220934,0.004629907,2.174838e-02,0.030274429,0.055151526
0.0119575996,0.162372723,0.1569437,0.3835065,0.5756705,0.3430122,0.1428215322,0.07028524,0.1149666,0.2370563,⋯,0.3097811,0.1721595,0.3246217,0.029285333,0.10579334,0.056024750,0.025634995,6.566806e-02,0.045767037,0.051110034
0.0503151915,0.058036260,0.1903953,0.3069982,0.6368655,0.3656543,0.1428970715,0.34050531,0.1132881,0.2508128,⋯,0.4668923,0.2035021,0.2894088,0.039070567,0.05407580,0.031152578,0.031286744,1.865517e-02,0.008259889,0.037288852
0.0002393563,0.121671767,0.2154224,0.5420303,0.5482209,0.3168748,0.0017748742,0.12076759,0.1189246,0.2557737,⋯,0.2653999,0.2194329,0.4657941,0.096742115,0.34204157,0.076021929,0.039869775,1.951875e-01,0.041538338,0.108998487
0.0220516172,0.438560460,0.2191368,0.4487820,0.5946929,0.3347930,0.0372078704,0.06253160,0.1189453,0.2673369,⋯,0.3038745,0.1816569,0.3120919,0.165516274,0.03693205,0.059791715,0.128191872,5.204657e-04,0.026915210,0.023631417


In [8]:
num_samples = nrow(p_est_draws) # Number of posterior samples
num_included_scenarios = nrow(included_scenarios) #Number of included scenarios to generate I^2 estimates for 
I_2_est_mat = array(-1, dim = c(num_included_scenarios, num_samples)) #Matrix of I^2 estimates (with each posterior sample corresponding to one I^2 estimate)
correction = 0.00001 #To prevent NaN in I2 estimates

#For each row corresponding to an included scenario
for(curr_row_ind in 1:num_included_scenarios){
    curr_row = included_scenarios[curr_row_ind, ]
    curr_scenario = curr_row$Scenario #Name of scenario to retrieve from outcome_df
    curr_scenario_ind = curr_row$ScenarioIndex #Index of Scenario so we know which value of p to get
    curr_outcomes = outcome_df %>% filter(Scenario == curr_scenario) #Get the relevant outcomes that are part of the scenario
    num_outcomes = nrow(curr_outcomes) #Get number of outcomes
    curr_p_vals = p_est_draws[ , curr_scenario_ind] #Get the values of p (column corresponding to the scenario index)
    
    curr_theta_indices = curr_outcomes %>% pull(ThetaIndex) #Get the theta index of each outcome. These should just be from 1:num_outcomes in the order of the outcome_df
    curr_n_vals = curr_outcomes %>% pull(N) #Sample size of each outcome in the given scenario
    curr_theta_vals = theta_est_draws[, curr_theta_indices] #matrix of theta values that match the outcomes for the given scenario
    
    #For each sample, we compute an estimate of I^2 (this means matching indices of p and thetas)
    I_2_est_samples = array(-1, dim = num_samples)
    for(curr_sample_ind in 1:num_samples){
        curr_p = curr_p_vals[curr_sample_ind] #Current p sample from the posterior distribution
        curr_theta = curr_theta_vals[curr_sample_ind, ] #Current theta samples from the posterior distribution
        
        without_het = array(-1, dim = length(num_outcomes)) #Between study variance in the absence of heterogeneity
        with_het = array(-1, dim = length(num_outcomes)) #Between study variance given heterogeneity
        
        for(i in 1:num_outcomes){
            without_het[i] = rbinom(n = 1, size = curr_n_vals[i], prob = curr_p) #For each outcome in the scenario, generate 1 random binomial draw using the outcome's sample size and the pooled estimate

            #We also generate 1 random binomial draw still using the outcome's sample size,
            #but this time using theta as the probability (includes random effect)
            with_het[i] = rbinom(n = 1, size = curr_n_vals[i], prob = curr_theta[i]) 
        }

        #Convert the binomial draws to proportions by dividing by the sample sizes
        without_het_props = without_het / curr_n_vals
        with_het_props = with_het / curr_n_vals

        #Compute the variance between proportions with and without heterogeneity - adding in the correction to both values to prevent divisions by 0 
        #which could occur for scenarios with only a small number of studies and/or smalller sample sizes
        var_het = var(with_het_props) + correction
        var_without_het = var(without_het_props) + correction
        I_2_est_samples[curr_sample_ind] = (var_het - var_without_het) / var_het
    }
    #Set the I^2 estimates for the scenario to the samples computed 
    I_2_est_mat[curr_row_ind, ] = I_2_est_samples
}

In [9]:
#Compute the I^2 estimate for each scenario by taking the median. We replace any values < 0 with 0
#These negative values can occur in scenarios with a small number of studies and/or small samples sizes or those with barely any heterogeneity. 
median_I2 = apply(I_2_est_mat, 1, median)
median_I2[median_I2 < 0] = 0 #Replace negative values with 0 
median_I2

[1] 0.9134498 0.9609659 0.9965774 0.9993254 0.9715538 0.9031414 0.8933926
 [8] 0.8231186 0.4361909 0.0000000 0.9891310 0.9022885 0.9367085

In [10]:
scenario_I2 = included_scenarios %>% mutate(I2_est = median_I2) 
output = scenario_df %>% left_join(scenario_I2 %>% select(Scenario, I2_est), by = "Scenario")#Create a DataFrame with scenario labels and the I^2 estimates to be output

In [11]:
saveRDS(output, file.path(output_dir, paste0("I2_Estimates_", sev_class_type, ".rds"))) #Save the output to RDS file for use in visualisations